<a href="https://colab.research.google.com/github/Marie5559/das172-examen2-pineda-sally/blob/main/Examen2_AeroCargo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sección nueva

# Sección nueva

In [1]:
%%writefile aerocargo.py
import numpy as np

def validar_matrices(cargas, capacidades):
    """
    Módulo 1: Validación y Coherencia Dimensional.
    Verifica que las matrices de cargas reales y capacidades máximas
    cumplan con los requisitos operativos.
    """
    # Convertimos los datos de entrada a arreglos de NumPy
    cargas_np = np.array(cargas)
    capacidades_np = np.array(capacidades)

    # 1. Verificar que ambas matrices sean regulares (bidimensionales)
    if cargas_np.ndim != 2 or capacidades_np.ndim != 2:
        return False

    # 2. Verificar que tengan exactamente las mismas dimensiones N x M
    if cargas_np.shape != capacidades_np.shape:
        return False

    N, M = cargas_np.shape

    # 3. Verificar que N >= 2 y M >= 2
    if N < 2 or M < 2:
        return False

    # 4. Verificar que las cargas sean >= 0 y las capacidades > 0
    if np.any(cargas_np < 0) or np.any(capacidades_np <= 0):
        return False

    return True

Writing aerocargo.py


In [3]:
%%writefile -a aerocargo.py

def calcular_ocupacion(cargas, capacidades):
    """
    Módulo 2: Cálculo de Ocupación y Detección de Sobrecarga.
    Calcula el porcentaje de ocupación por celda e identifica
    las coordenadas con sobrecarga (>100%).
    """
    cargas_np = np.array(cargas)
    capacidades_np = np.array(capacidades)

    # Generar matriz de porcentajes de ocupación
    porcentajes = (cargas_np / capacidades_np) * 100.0

    # Identificar coordenadas de las celdas en sobrecarga (> 100.0%)
    filas, columnas = np.where(porcentajes > 100.0)
    coordenadas_sobrecarga = list(zip(filas.tolist(), columnas.tolist()))

    # Retornar estructura con matriz y lista de coordenadas
    return {
        "matriz_porcentajes": porcentajes,
        "coordenadas_criticas": coordenadas_sobrecarga
    }

Appending to aerocargo.py


In [5]:
%%writefile -a aerocargo.py

def evaluar_balance(cargas, tolerancia):
    """
    Módulo 3: Evaluación de Balance y Simetría.
    Calcula los pesos longitudinales y evalúa el desbalance lateral
    omitiendo el eje central si el número de columnas es impar.
    """
    cargas_np = np.array(cargas)

    # 1. Vector de peso total por cada fila longitudinal (eje 1 suma las columnas de cada fila)
    pesos_longitudinales = np.sum(cargas_np, axis=1).tolist()

    # 2. División de la matriz para el balance lateral
    M = cargas_np.shape[1]
    mitad = M // 2

    if M % 2 == 0:
        # Si es par, se divide exactamente a la mitad
        suma_izquierda = np.sum(cargas_np[:, :mitad])
        suma_derecha = np.sum(cargas_np[:, mitad:])
    else:
        # Si es impar, se omite la columna central
        suma_izquierda = np.sum(cargas_np[:, :mitad])
        suma_derecha = np.sum(cargas_np[:, mitad+1:])

    # 3. Cálculo del desbalance absoluto
    desbalance_lateral = abs(suma_izquierda - suma_derecha)

    # 4. Evaluación contra el umbral de tolerancia
    estado_balance = bool(desbalance_lateral <= tolerancia)

    return {
        "pesos_longitudinales": pesos_longitudinales,
        "desbalance_lateral": float(desbalance_lateral),
        "estado_balance": estado_balance
    }

Appending to aerocargo.py


In [ ]:
%%writefile -a aerocargo.py

def extraer_submatriz_critica(porcentajes, k, p):
    """
    Módulo 4: Extracción de Submatriz de Sobrecarga Crítica.
    Recorre submatrices contiguas de tamaño k x p para encontrar
    la que presente el mayor promedio de ocupación.
    """
    porcentajes_np = np.array(porcentajes)
    N, M = porcentajes_np.shape

    # Validar que la ventana k x p no sea más grande que el piso completo
    if k > N or p > M:
        return None

    max_promedio = -1.0
    submatriz_critica = None

    # Recorrer la matriz fila por fila y columna por columna
    for i in range(N - k + 1):
        for j in range(M - p + 1):
            # Extraer la submatriz actual usando "slicing" (rebanado) de NumPy
            submatriz_actual = porcentajes_np[i:i+k, j:j+p]

            # Calcular el promedio de ocupación de esa zona específica
            promedio_actual = np.mean(submatriz_actual)

            # Si este promedio es el más alto hasta ahora, lo guardamos
            if promedio_actual > max_promedio:
                max_promedio = promedio_actual
                submatriz_critica = submatriz_actual

    # Retornamos la submatriz extraída convirtiéndola de nuevo a lista normal
    return submatriz_critica.tolist()